Reproduces the SSG++ results of the paper and lets you run your own experiments. Run it from the repository root, starting with the imports.

* **Reproduce Tables 1-3**: prints the SSG++ columns of Tables 1, 2 and 3
* **Run your own experiment**: one dataset at a lambda, mu, t and T of your choice
* **Regularization path**: a lambda grid up to the first perfectly balanced solution
* **Figure 2**: `cosponsorBel/sweep_cosponsorBel_mu_sensitivity_{density,representation}.pdf` and `cosponsorBel/sweep_cosponsorBel_legend.pdf`
* **Figure 3**: `bel_subgraph.pdf` ($\lambda = 0$), `bel_subgraph_Div.pdf` ($\lambda = 100$) and `bel_legend.pdf`

## Import required packages

In [ ]:
import os
import sys
import copy
import numpy as np
import networkx as nx

from super_greedy_set_linear import *
import utils
import matplotlib.pyplot as plt 
import matplotlib
%matplotlib inline 
import pandas as pd
from init_graph import init_graph

import matplotlib.patches as mpatches


from tqdm import tqdm


# Import seaborn
import seaborn as sns

# Apply the default theme
sns.set_theme()

plt.rcParams["figure.dpi"]=100
plt.rcParams['savefig.dpi']=300


## Reproduce Tables 1-3
Runs SSG++ with the settings of the paper ($\mu = 2$, $\alpha$ at its bound, $t = 5$ SuperGreedy++ passes, $T = 5$ outer iterations, fixed marginal gains) at $\lambda = 0$, the densest subgraph of Table 2, and at the first perfectly balanced $\lambda$ of Table 3. All four datasets take about five minutes; shorten `DATASETS` to run fewer. The DDSP columns of Table 3 were computed with the implementation of Miyauchi et al. (KDD 2023) and are not reproduced here.

In [ ]:
import contextlib, io, time

def quiet(f, *args, **kwargs):
    """Call f without its progress messages."""
    with contextlib.redirect_stdout(io.StringIO()):
        return f(*args, **kwargs)

def solve(G, groups, lam, mu=2.0, num_passes=5, num_outer=5, lin_method=1):
    """SSG++ from a cold start (the whole graph); alpha=None puts alpha at its bound."""
    return quiet(ssg_pp, G, G, groups, lam=lam, mu=mu, alpha=None,
                 num_passes=num_passes, num_outer=num_outer, lin_method=lin_method)[0]

def evaluate(G, groups, S):
    """Size, density rho = 2e/|S|, minimum represented proportion, r_adj and vertices per group of S."""
    H = G.subgraph(S)
    sizes = [len(set(S) & set(g)) for g in groups]
    return {"size": len(S), "rho": 2 * H.number_of_edges() / len(S), "min_prop": min(sizes) / len(S),
            "r_adj": utils.compute_adjusted_assortativity(H, groups), "group_sizes": sizes}

# first perfectly balanced lambda: on the grid linspace(0, 5000, 496) for Bel and BlogCatalog,
# the values reported in the paper for PubMed and Oklahoma
PAPER_LAMBDA = {"cosponsorBel": 90.9090909090909, "blogcatalog": 4353.535353535353, "pubmed": 100.0, "oklahoma": 1400.0}
DATASETS = ["cosponsorBel", "blogcatalog", "pubmed", "oklahoma"]

rows = {}
for ds in DATASETS:
    t0 = time.time()
    G, groups = quiet(init_graph, ds)
    rows[ds] = {"n": G.number_of_nodes(), "m": G.number_of_edges(), "L": len(groups),
                "sizes": [sum(v in G for v in g) for g in groups],
                "rho": 2 * G.number_of_edges() / G.number_of_nodes(),
                "r_adj": utils.compute_adjusted_assortativity(G, groups),
                "dsg": evaluate(G, groups, solve(G, groups, 0.0)),
                "bal": evaluate(G, groups, solve(G, groups, PAPER_LAMBDA[ds]))}
    print(f"{ds}: done in {time.time() - t0:.0f} s")

print("\nTable 1: datasets")
print(f"{'dataset':13s} {'n':>6s} {'m':>7s} {'L':>3s} {'min/max |S_l|':>14s} {'rho(G)':>7s} {'r_adj(G)':>8s}")
for ds, r in rows.items():
    print(f"{ds:13s} {r['n']:6d} {r['m']:7d} {r['L']:3d} {min(r['sizes']):>8d}/{max(r['sizes']):<5d} {r['rho']:7.2f} {r['r_adj']:8.2f}")
print("\nTable 2: densest subgraph (lambda = 0)")
print(f"{'dataset':13s} {'|S|':>5s} {'rho(S)':>7s} {'r_adj(S)':>8s} {'min prop':>9s}")
for ds, r in rows.items():
    d = r["dsg"]
    print(f"{ds:13s} {d['size']:5d} {d['rho']:7.2f} {d['r_adj']:8.2f} {d['min_prop']:9.3f}")
print("\nTable 3: SSG++ at perfect balance")
print(f"{'dataset':13s} {'lambda':>8s} {'|S|':>5s} {'rho(S)':>7s} {'r_adj(S)':>8s}  vertices per group")
for ds, r in rows.items():
    b = r["bal"]
    per_group = f"{b['group_sizes'][0]} each" if len(set(b["group_sizes"])) == 1 else str(b["group_sizes"])
    print(f"{ds:13s} {PAPER_LAMBDA[ds]:8.2f} {b['size']:5d} {b['rho']:7.2f} {b['r_adj']:8.2f}  {per_group}")


## Run your own experiment
Choose a dataset and the parameters, and run the cell. $\lambda = 0$ gives the densest subgraph; larger $\lambda$ gives more balanced subgraphs. Uses `quiet`, `solve` and `evaluate` from the cell above.

In [ ]:
dataset = "cosponsorBel"       # "cosponsorBel", "blogcatalog", "pubmed" or "oklahoma"
lam = 50.0                     # lambda >= 0
mu = 2.0                       # LSE temperature
num_passes, num_outer = 5, 5   # SuperGreedy++ passes t and SSG++ outer iterations T
lin_method = 1                 # 1: fixed marginal gains (as in the paper), 2: chain modularization

G, groups = quiet(init_graph, dataset)
r = evaluate(G, groups, solve(G, groups, lam, mu, num_passes, num_outer, lin_method))
print(f"{dataset}, lambda = {lam:g}: |S| = {r['size']}, rho(S) = {r['rho']:.3f}, r_adj(S) = {r['r_adj']:.3f}")
print(f"min. represented proportion = {r['min_prop']:.4f} (perfect balance: {1 / len(groups):.4f})")
print("vertices per group:", r["group_sizes"])


### Regularization path
Solves the lambdas of a grid in increasing order until the first perfectly balanced solution, which is how the balanced $\lambda$ of Table 3 was found for Bel and BlogCatalog. Bel takes seconds, BlogCatalog about two hours.

In [ ]:
dataset = "cosponsorBel"
lams = np.linspace(0, 5000, 496)

G, groups = quiet(init_graph, dataset)
print(f"{'lambda':>9s} {'|S|':>6s} {'rho(S)':>8s} {'min prop':>9s} {'r_adj(S)':>9s}")
for lam in lams:
    r = evaluate(G, groups, solve(G, groups, float(lam)))
    print(f"{lam:9.2f} {r['size']:6d} {r['rho']:8.3f} {r['min_prop']:9.4f} {r['r_adj']:9.3f}")
    if r["min_prop"] >= 1 / len(groups) - 1e-9:
        print("first perfectly balanced solution")
        break


## Figure 2: regularization path of Bel for varying $\mu$

In [ ]:
import matplotlib.lines as mlines

dataset_name = "cosponsorBel"

mu_colors = ["tab:orange","tab:blue", "tab:green", "tab:purple"]

muVec = [0.5,1,2,5]

handles = [mlines.Line2D([], [], color=mu_colors[i], linewidth=2, label=rf"$\mu = {mu}$")
           for i, mu in enumerate(muVec)]
handles.append(mlines.Line2D([], [], color="tab:red", linestyle="dashed", linewidth=2, label=r"$1/L$"))

fig_leg = plt.figure(figsize=(16, 0.6))
leg = fig_leg.legend(
    handles=handles,
    loc='center',
    ncol=len(handles),          # one row
    frameon=False,
    fontsize=16,
    handlelength=2.0,
    handletextpad=0.5,
    columnspacing=2.0,
)
fig_leg.canvas.draw()
bbox = leg.get_window_extent().transformed(fig_leg.dpi_scale_trans.inverted())
fig_leg.savefig(f"{dataset_name}/sweep_{dataset_name}_legend.pdf",
                format='pdf', bbox_inches=bbox, pad_inches=0.02)
plt.show()

In [ ]:
L = 8
dataset_name = 'cosponsorBel'
lin_method = 1
G,protected_nodes = init_graph(dataset_name)
lam_vec = np.linspace(0,200.0,300)
L_vec = [1.0/L for _ in range(len(lam_vec))]
colours = ["tab:orange","tab:blue", "tab:green", "tab:purple"]
muVec = [0.5,1.0,2.0,5.0]

densityvec_collection = []
protected_portion_in_sub_vec_collection = []
for mu in muVec:
    density_vec = []
    protected_portion_in_sub_vec = []
    for lam in lam_vec:
        super_greedy_pp_R = ssg_pp(G, G, protected_nodes_all=protected_nodes, lam=lam, mu=mu, alpha=None, num_passes=5,num_outer=10, lin_method=lin_method)
        density_vec.append(utils.compute_density(super_greedy_pp_R[0], G))
        protected_portion_in_sub_vec.append(np.min([len(set(super_greedy_pp_R[0]).intersection(set(prot_nodes)))/len(super_greedy_pp_R[0]) for prot_nodes in protected_nodes]))
    densityvec_collection.append(density_vec)
    protected_portion_in_sub_vec_collection.append(protected_portion_in_sub_vec)

## Plotting the results for different mu values
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(5, 5))
# ── Left: Minimum represented proportion ──
ax1.set_facecolor('#ffffff')
for i, mu in enumerate(muVec):
    #increase the size of the axis labels and ticks
    ax1.tick_params(axis='both', which='major', labelsize=14)
    ax1.plot((lam_vec), protected_portion_in_sub_vec_collection[i], label=f"SSP++ linearization (mu={mu})", color=colours[i])
ax1.plot((lam_vec), L_vec, color="tab:red", linestyle="dashed", label="1/L")
ax1.set_xlabel(r"$\lambda$",fontsize=20)
ax1.set_ylabel(r"$\underset{\ell}{\mathrm{min}} \; \dfrac{|\mathcal{S} \cap \mathcal{S}_{\ell}|}{|\mathcal{S}|}$",fontsize=20)
#ax1.legend(fontsize=18)
# ── Right: Density ──
ax2.set_facecolor('#ffffff')
for i, mu in enumerate(muVec):
    ax2.plot((lam_vec), [2*densityvec_collection[i][j] for j in range(len(densityvec_collection[i]))], label=f"SSP++ linearization (mu={mu})", color=colours[i])
ax2.set_xlabel(r"$\lambda$",fontsize=20)
ax2.tick_params(axis='both', which='major', labelsize=14)
ax2.set_ylabel(r"$\rho(\mathcal{S})$",fontsize=20)
#ax2.legend(fontsize=18)
plt.tight_layout()
plt.savefig(f"{dataset_name}/sweep_{dataset_name}_mu_sensitivity.pdf", format='pdf', bbox_inches='tight')
plt.show()







In [ ]:
## Plotting the results for different mu values

# ── Figure 1: Minimum represented proportion ──
fig1, ax1 = plt.subplots(figsize=(8, 6))
ax1.set_facecolor('#ffffff')
ax1.tick_params(axis='both', which='major', labelsize=22)
for i, mu in enumerate(muVec):
    ax1.plot(lam_vec, protected_portion_in_sub_vec_collection[i],
             label=f"SSP++ linearization (mu={mu})", color=colours[i])
ax1.plot(lam_vec, L_vec, color="tab:red", linestyle="dashed", label="1/L")
ax1.set_xlabel(r"$\lambda$", fontsize=22)
ax1.set_ylabel(r"$\underset{\ell}{\mathrm{min}} \; {|\mathcal{S} \cap \mathcal{S}_{\ell}|}/{|\mathcal{S}|}$", fontsize=22)
#ax1.legend(fontsize=18)
fig1.tight_layout()
fig1.savefig(f"{dataset_name}/sweep_{dataset_name}_mu_sensitivity_representation.pdf",
             format='pdf', bbox_inches='tight')

# ── Figure 2: Density ──
fig2, ax2 = plt.subplots(figsize=(8, 6))
ax2.set_facecolor('#ffffff')
ax2.tick_params(axis='both', which='major', labelsize=22)
for i, mu in enumerate(muVec):
    ax2.plot(lam_vec, [2*d for d in densityvec_collection[i]],
             label=f"SSP++ linearization (mu={mu})", color=colours[i])
ax2.set_xlabel(r"$\lambda$", fontsize=22)
ax2.set_ylabel(r"$\rho(\mathcal{S})$", fontsize=22)
#ax2.legend(fontsize=18)
fig2.tight_layout()
fig2.savefig(f"{dataset_name}/sweep_{dataset_name}_mu_sensitivity_density.pdf",
             format='pdf', bbox_inches='tight')

plt.show()

## Figure 3: the Belgian Chamber of Representatives

In [ ]:
G_bel, protected_nodes_bel = init_graph("cosponsorBel")
pos_bel = nx.spring_layout(G_bel, seed=42)
# ── Party family colours (one colour per family, shape distinguishes language) ─
family_colors = {
    "Green":       "#4DAF4A",
    "socialist":   "#E41A1C",
    "Centrist":        "#FF7F00",
    "Liberal":     "#377EB8",
    "Center right": "#FFFF33",
    "farright":    "#984EA3",
    "other":       "#888888",
}
 
party_to_family = {
    "ECOLO":        "Green",
    "SOC-F":        "socialist",
    "SOC-V":        "socialist",
    "CDEM-F":       "Centrist",
    "CDEM-V":       "Centrist",
    "LDD":          "farright",
    "CDEM-V/VOLKS": "Centrist",
    "VOLKS":        "Center right",
    "LIB-F":        "Liberal",
    "LIB-V":        "Liberal",
    "PP":           "farright",
    "DLB":          "other",
    "FN":           "other",
    "IND":          "other",
}
 
french_parties = { "SOC-F", "CDEM-F", "LIB-F", "FN", "PP", "DLB"}
dutch_parties  = {"SOC-V", "CDEM-V", "LIB-V", "LDD", "CDEM-V/VOLKS", "VOLKS"}
 
def bel_family_color(node):
    party = G_bel.nodes[node].get('party', '')
    return family_colors.get(party_to_family.get(party, 'other'), '#CCCCCC')
 
def bel_marker(node):
    party = G_bel.nodes[node].get('party', '')
    if party in french_parties:
        return 'o'
    elif party in dutch_parties:
        return 's'
    return 'D'
 
def bel_edgecolor(node):
    party = G_bel.nodes[node].get('party', '')
    return '#333333' if party in dutch_parties else 'none'
 
def draw_nodes_by_marker(G, pos, nodelist, alpha, node_size, ax):
    """Draw nodes grouped by marker shape."""
    groups = {}
    for n in nodelist:
        m = bel_marker(n)
        groups.setdefault(m, []).append(n)
    for marker, nodes in groups.items():
        nx.draw_networkx_nodes(
            G, pos, nodelist=nodes,
            node_color=[bel_family_color(n) for n in nodes],
            edgecolors=[bel_edgecolor(n) for n in nodes],
            linewidths=1.2, node_shape=marker,
            node_size=node_size, alpha=alpha, ax=ax
        )
 
def bel_legend(parties):
    import matplotlib.patches as mpatches
    import matplotlib.lines as mlines
    handles = []
    for fam in sorted({party_to_family.get(p, 'other') for p in parties}):
        handles.append(mpatches.Patch(color=family_colors[fam], label=fam.capitalize()))
    handles.append(mlines.Line2D([], [], color='none', label=''))
    handles.append(mlines.Line2D([], [], marker='o', color='gray', linestyle='None',
                                  markersize=8, label='French-speaking'))
    handles.append(mlines.Line2D([], [], marker='s', color='gray', linestyle='None',
                                  markeredgecolor='#333333', markeredgewidth=1.2,
                                  markersize=8, label='Dutch-speaking'))
    handles.append(mlines.Line2D([], [], marker='D', color='gray', linestyle='None',
                              markersize=8, label='National/Bilingual'))
    return handles
 
 #print each party and its number of nodes in the original graph:
print("Nodes per party in original graph:", {p: sum(1 for n in G_bel.nodes() if G_bel.nodes[n].get('party') == p) for p in party_to_family.keys()})
# ── Full graph ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_facecolor("#ffffff")
ax.set_title("Belgian Cosponsor Network – Legislature 54 (2014–2019)", fontsize=25)
for spine in ax.spines.values():
    spine.set_visible(False)
draw_nodes_by_marker(G_bel, pos_bel, list(G_bel.nodes()), alpha=0.85, node_size=300, ax=ax)
nx.draw_networkx_edges(G_bel, pos_bel, alpha=0.15, ax=ax)
present_parties = sorted({G_bel.nodes[n].get('party') for n in G_bel.nodes()} & party_to_family.keys())
ax.legend(handles=bel_legend(present_parties), loc='upper right', fontsize=20, ncol=1)
plt.tight_layout()
plt.savefig("bel_full.svg", format='svg', bbox_inches='tight')
plt.show()
 
 
# ── Subgraph at chosen λ ────────────────────────────────────────────────────
_, protected_nodes_bel = init_graph("cosponsorBel")
lam_bel = 100   # ← adjust lambda here
result_bel = ssg_pp(
    G_bel, G_bel, protected_nodes_bel,
    lam=lam_bel, mu=2.0, alpha=None, num_passes=10, num_outer=20
)
S_bel = G_bel.subgraph(result_bel[0])
 
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_facecolor("#ffffff")
#ax.set_title(f"Belgian Cosponsor Network – Dense Subgraph (λ = {lam_bel})", fontsize=25)
for spine in ax.spines.values():
    spine.set_visible(False)
# Non-selected: faded
non_selected = [n for n in G_bel.nodes() if n not in S_bel]
draw_nodes_by_marker(G_bel, pos_bel, non_selected, alpha=0.35, node_size=250, ax=ax)
nx.draw_networkx_edges(G_bel, pos_bel, alpha=0.08, ax=ax)
# Selected: full colour
draw_nodes_by_marker(G_bel, pos_bel, list(S_bel.nodes()), alpha=0.95, node_size=300, ax=ax)
nx.draw_networkx_edges(S_bel, pos_bel, alpha=1, ax=ax)
 
density_bel =  2*S_bel.number_of_edges() / max(1, S_bel.number_of_nodes())
#ax.text(0.01, 0.02, f"Density: {density_bel:.3f}  |  Nodes: {S_bel.number_of_nodes()}",
#        transform=ax.transAxes, fontsize=20)
#sub_parties = sorted({G_bel.nodes[n].get('party') for n in G_bel.nodes()})
#ax.legend(handles=bel_legend(sub_parties), loc='upper right', fontsize=18, ncol=1)
plt.tight_layout()
plt.savefig("bel_subgraph_Div.pdf", format='pdf', bbox_inches='tight')
plt.show()
 
print(f"Subgraph: {S_bel.number_of_nodes()} nodes, {S_bel.number_of_edges()} edges, density {density_bel:.4f}")
print("Nodes per party:", {p: sum(1 for n in S_bel.nodes() if G_bel.nodes[n].get('party') == p)
                           for p in present_parties})
 

In [ ]:
# ── Subgraph at chosen λ ────────────────────────────────────────────────────
_, protected_nodes_bel = init_graph("cosponsorBel")
lam_bel = 0   # ← adjust lambda here
result_bel = ssg_pp(
    G_bel, G_bel, protected_nodes_bel,
    lam=lam_bel, mu=2.0, alpha=None, num_passes=10, num_outer=20
)
S_bel = G_bel.subgraph(result_bel[0])
 
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_facecolor("#ffffff")
#ax.set_title(f"Belgian Cosponsor Network – Dense Subgraph (λ = {lam_bel})", fontsize=25)
for spine in ax.spines.values():
    spine.set_visible(False)
# Non-selected: faded
non_selected = [n for n in G_bel.nodes() if n not in S_bel]
draw_nodes_by_marker(G_bel, pos_bel, non_selected, alpha=0.35, node_size=250, ax=ax)
nx.draw_networkx_edges(G_bel, pos_bel, alpha=0.08, ax=ax)
# Selected: full colour
draw_nodes_by_marker(G_bel, pos_bel, list(S_bel.nodes()), alpha=0.95, node_size=300, ax=ax)
nx.draw_networkx_edges(S_bel, pos_bel, alpha=1, ax=ax)
 
density_bel =  2*S_bel.number_of_edges() / max(1, S_bel.number_of_nodes())
#ax.text(0.01, 0.02, f"Density: {density_bel:.3f}  |  Nodes: {S_bel.number_of_nodes()}",
#        transform=ax.transAxes, fontsize=20)
#sub_parties = sorted({G_bel.nodes[n].get('party') for n in G_bel.nodes()})
#ax.legend(handles=bel_legend(sub_parties), loc='upper right', fontsize=18, ncol=1)
plt.tight_layout()
plt.savefig("bel_subgraph.pdf", format='pdf', bbox_inches='tight')
plt.show()
 
print(f"Subgraph: {S_bel.number_of_nodes()} nodes, {S_bel.number_of_edges()} edges, density {density_bel:.4f}")
print("Nodes per party:", {p: sum(1 for n in S_bel.nodes() if G_bel.nodes[n].get('party') == p)
                           for p in present_parties})
 

In [ ]:
all_parties = sorted({G_bel.nodes[n].get('party') for n in G_bel.nodes()} & party_to_family.keys())
# drop the empty spacer handle so colours and shapes follow on each other
handles = [h for h in bel_legend(all_parties) if h.get_label() != '']

# matplotlib fills legends column-first; reorder so the grid reads row-wise
ncol, nrow = 4, 2
rows = [handles[r*ncol:(r+1)*ncol] for r in range(nrow)]
handles_grid = [rows[r][c] for c in range(ncol) for r in range(nrow) if c < len(rows[r])]

fig_leg = plt.figure(figsize=(12, 1.0))
leg = fig_leg.legend(
    handles=handles_grid,
    loc='center',
    ncol=ncol,                  # 4 columns x 2 rows
    frameon=False,
    fontsize=14,
    handletextpad=0.4,
    columnspacing=1.6,
)
fig_leg.canvas.draw()
bbox = leg.get_window_extent().transformed(fig_leg.dpi_scale_trans.inverted())
fig_leg.savefig("bel_legend.pdf", format='pdf', bbox_inches=bbox, pad_inches=0.02)
plt.show()